# Financial News Sentiment Model
Run the cells in order (Runtime > Run all). Results come from a held-out test set.

In [ ]:
!pip install -q scikit-learn pandas matplotlib datasets yfinance

In [ ]:
%%writefile news_sentiment.py
"""
Financial news headline sentiment model (bearish / neutral / bullish)
+ optional link between daily sentiment and next-day stock returns.

Run in Google Colab or locally:
    pip install scikit-learn pandas matplotlib datasets yfinance
    python news_sentiment.py                      # downloads public dataset from Hugging Face
    python news_sentiment.py --csv my_data.csv    # or use your own CSV with columns: text,label
    python news_sentiment.py --prices headlines.csv --ticker RELIANCE.NS
        (headlines.csv columns: date,text  -> measures sentiment vs next-day return)

All numbers printed come from a held-out test set the model never trained on.
"""
import argparse
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

LABELS = ["bearish", "bullish", "neutral"]  # 0, 1, 2 in zeroshot/twitter-financial-news-sentiment
SEED = 42


def load_data(csv_path=None):
    """Return (train_df, test_df) with columns text,label (label in 0/1/2 or strings)."""
    if csv_path:
        df = pd.read_csv(csv_path)[["text", "label"]].dropna()
        if df["label"].dtype == object:
            df["label"] = df["label"].str.lower().map({l: i for i, l in enumerate(LABELS)})
        df = df.dropna()
        df["label"] = df["label"].astype(int)
        train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
        return train, test
    from datasets import load_dataset
    ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
    train = ds["train"].to_pandas()[["text", "label"]]
    test = ds["validation"].to_pandas()[["text", "label"]]
    return train, test


def train_and_eval(train, test):
    # Hold out part of train for model selection; test set is touched exactly once, at the end.
    tr, val = train_test_split(train, test_size=0.15, stratify=train["label"], random_state=SEED)
    best, best_f1 = None, -1
    for C in [0.3, 1, 3, 10]:
        pipe = make_pipeline(
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True),
            LogisticRegression(C=C, max_iter=2000, class_weight="balanced"),
        )
        pipe.fit(tr["text"], tr["label"])
        f1 = f1_score(val["label"], pipe.predict(val["text"]), average="macro")
        print(f"C={C:<4} validation macro-F1 = {f1:.3f}")
        if f1 > best_f1:
            best, best_f1 = pipe, f1
    # Refit chosen config on all training data, then evaluate once on the test set.
    best.fit(train["text"], train["label"])
    pred = best.predict(test["text"])

    # Majority-class baseline, so accuracy has context.
    majority = train["label"].mode()[0]
    base_acc = accuracy_score(test["label"], np.full(len(test), majority))

    print("\n=== TEST SET RESULTS (held out) ===")
    print(f"Majority-class baseline accuracy: {base_acc:.3f}")
    print(f"Model accuracy:                   {accuracy_score(test['label'], pred):.3f}")
    print(f"Model macro-F1:                   {f1_score(test['label'], pred, average='macro'):.3f}")
    print(classification_report(test["label"], pred, target_names=LABELS, digits=3))
    cm = confusion_matrix(test["label"], pred)
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues")
    plt.title("Headline sentiment - confusion matrix (test set)")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.close()
    return best


def sentiment_vs_returns(model, headlines_csv, ticker):
    """Daily sentiment score = mean(P(bullish) - P(bearish)); compare with NEXT-day return."""
    import yfinance as yf
    h = pd.read_csv(headlines_csv)
    h["date"] = pd.to_datetime(h["date"]).dt.normalize()
    proba = model.predict_proba(h["text"])
    h["score"] = proba[:, 1] - proba[:, 0]
    daily = h.groupby("date")["score"].mean()

    px = yf.download(ticker, start=daily.index.min() - pd.Timedelta(days=5),
                     end=daily.index.max() + pd.Timedelta(days=7), auto_adjust=True, progress=False)
    close = px["Close"].squeeze()
    ret_next = close.pct_change().shift(-1).rename("next_day_return")  # no look-ahead: score at close of day t vs return t->t+1
    df = pd.concat([daily, ret_next], axis=1, join="inner").dropna()
    print(f"\n=== Sentiment vs next-day return for {ticker} ({len(df)} days) ===")
    if len(df) < 30:
        print("Fewer than 30 days: any correlation here is not reliable.")
    print(f"Pearson correlation:  {df['score'].corr(df['next_day_return']):.3f}")
    print(f"Spearman correlation: {df['score'].corr(df['next_day_return'], method='spearman'):.3f}")
    hit = ((df["score"] > 0) == (df["next_day_return"] > 0)).mean()
    print(f"Direction hit-rate:   {hit:.3f}  (50% = coin flip)")
    plt.scatter(df["score"], df["next_day_return"] * 100, alpha=0.6)
    plt.xlabel("Daily sentiment score"); plt.ylabel("Next-day return (%)")
    plt.title(f"{ticker}: sentiment vs next-day return")
    plt.savefig("sentiment_vs_return.png", dpi=150, bbox_inches="tight")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv")
    ap.add_argument("--prices", help="CSV with date,text columns for the returns analysis")
    ap.add_argument("--ticker", default="^NSEI")
    a = ap.parse_args()
    train, test = load_data(a.csv)
    print(f"train={len(train)}  test={len(test)}")
    model = train_and_eval(train, test)
    if a.prices:
        sentiment_vs_returns(model, a.prices, a.ticker)


In [ ]:
!python news_sentiment.py

In [ ]:
from IPython.display import Image
Image('confusion_matrix.png')